# Phase 0: Fundamentals
## Externalization in LLM Agents — Zhou et al., 2026

> **Goal:** Understand the paper's core ideas before writing a single line of code.
> No implementations here — only concepts, diagrams, and intuition.

---

## 1. The Problem: Why LLM Agents Fail

Current LLM agents fail in predictable, structural ways — not because the models aren't smart enough, but because they're asked to carry too much:

| What the agent carries | What breaks |
|------------------------|-------------|
| User history in context window | Forgotten after session ends |
| Procedures re-derived each call | Inconsistent execution, hallucinated steps |
| Ad-hoc API calls | Brittle, untyped, no error recovery |
| Coordination logic in the prompt | No gates, no observability, no rollback |

The paper's diagnosis: **these burdens shouldn't be inside the model at all.**

## 2. The Core Idea: Externalization

**Externalization** = deliberately moving cognitive burdens from inside the model (weights + context) into persistent, inspectable, reusable external structures.

```
BEFORE externalization:
┌─────────────────────────────────────────────┐
│                 LLM MODEL                   │
│  history + procedures + APIs + coordination │
│         Everything crammed in here          │
└─────────────────────────────────────────────┘
         ↑ fragile, forgetful, inconsistent

AFTER externalization:
┌───────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  MEMORY   │    │  SKILLS  │    │PROTOCOLS │    │ HARNESS  │
│ DynamoDB  │    │ SKILL.md │    │   MCP    │    │  Strands │
└─────┬─────┘    └────┬─────┘    └────┬─────┘    └────┬─────┘
      └───────────────┴───────────────┴───────────────┘
                              │
                      ┌───────┴───────┐
                      │   LLM MODEL   │
                      │  (reasoning   │
                      │   only)       │
                      └───────────────┘
              ↑ each burden in the right place
```

The model stays fixed. The external infrastructure changes everything.

## 3. The Four Externalizations

### 3.1 Memory — §3

**Problem:** LLMs have no persistent memory across sessions. Users re-explain themselves every time.

**Solution:** A 4-tier memory architecture stored in DynamoDB:

```
WORKING MEMORY    → What is the agent doing RIGHT NOW?
                    Session intent, active tool calls, pending approvals
                    TTL: 24 hours

EPISODIC MEMORY   → What HAS HAPPENED in the past?
                    Completed tasks, decisions made, outcomes
                    TTL: 90 days  

SEMANTIC MEMORY   → What does the agent KNOW about the domain?
                    Stable facts, heuristics, market concepts
                    TTL: permanent

PERSONAL MEMORY   → Who is the USER?
                    Risk tolerance, goals, preferences, patterns
                    TTL: permanent
```

**Key insight:** This converts *recall* (expensive, unreliable, from model weights) into *recognition* (cheap, reliable, from a curated database retrieve).

**Episodic → Semantic promotion:** When the same episodic pattern recurs many times (e.g., user always checks NVDA before trading), it gets promoted to semantic memory — the agent "learns" without fine-tuning.

### 3.2 Skills — §4

**Problem:** LLMs re-derive *how* to do things every call — inconsistently, sometimes hallucinating steps.

**Solution:** Package procedures into **SKILL.md files** stored in S3. The agent loads them at runtime.

A skill file contains:
- Capability boundary (what it does and doesn't do)
- Preconditions (what must be true before running)
- Operational procedure (step-by-step, mandatory)
- Decision heuristics (edge case handling)
- Normative constraints (what the agent must NEVER do)

**Progressive Disclosure (§4.3.3):** Skills are loaded in stages to manage context budget:

```
Stage 1 (always loaded):     trade_execution — executes stock trades
Stage 2 (skill selected):    + preconditions, scope, constraints
Stage 3 (skill executing):   + full 8-step procedure, examples, tool bindings
```

Only the active skill's Stage 3 is in context. Others stay at Stage 1 — saving thousands of tokens.

### 3.3 Protocols — §5

**Problem:** Ad-hoc API calls are brittle, untyped, and fail silently. The agent can't reason about errors.

**Solution:** **MCP (Model Context Protocol)** — every external service interaction is a typed, schema-validated contract.

Every MCP tool returns the same envelope:
```json
{
  "tool": "alpaca_get_positions",
  "status": "success | error | rate_limited | timeout",
  "data": { "...typed payload..." },
  "metadata": {
    "timestamp": "2026-06-18T15:00:00Z",
    "latency_ms": 142,
    "source": "alpaca_v2_api"
  },
  "error": null
}
```

The agent can branch on `status`, knows the data shape, and has retry guidance — instead of crashing on a raw HTTP error.

### 3.4 Harness — §6

**Problem:** No mechanism to sequence, gate, observe, or recover multi-step workflows. Agents either act without oversight or are blocked by excessive approval requirements.

**Solution:** The harness is the **unification layer** — it does not add intelligence, it governs execution.

```
HARNESS LOOP (every agent invocation):

  1. RETRIEVE MEMORY
     Personal profile + working context + top-3 episodic + relevant semantic
     → Converts recall to recognition

  2. SELECT SKILL + PROGRESSIVE DISCLOSURE
     Route intent to the right skill, load Stage 2 (constraints first)
     → Agent knows what NOT to do before it plans

  3. REASON (Bedrock)
     Model reasons within the scaffolding — memory + skill + tools
     → Model focuses only on reasoning, not coordination

  4. EXECUTE MCP TOOLS
     Typed calls, structured responses, logged to CloudWatch
     → Observable, debuggable, retryable

  5. APPROVAL GATE (if irreversible action)
     Halt → SNS → user confirms → resume or discard
     → Human stays in the loop for financial actions

  6. WRITE EPISODIC MEMORY
     What happened, what skill, what outcome
     → Agent learns from each interaction

  7. RESPOND
```

**Tiered approval (§6.2.3):** Not all actions need the same gate:
- Read-only queries → no gate
- Small trades (< $500) → inline 10-second confirmation banner
- Large trades (≥ $500) → SNS email/SMS, 5-minute window

## 4. Context Budget Management (§6.2.6)

The harness explicitly manages token allocation — context is a finite resource:

```
Total context budget: 100,000 tokens (Claude 3.5 Sonnet)

Allocation:
  System prompt + active skill (Stage 3):   ~3,000 tokens   (3%)
  PERSONAL memory (user profile):           ~1,000 tokens   (1%)
  Top-3 EPISODIC memories:                  ~2,000 tokens   (2%)
  SEMANTIC knowledge:                       ~1,500 tokens   (1.5%)
  Current conversation:                     ~5,000 tokens   (5%)
  Tool call results:                        ~8,000 tokens   (8%)
  Reserved for reasoning:                  ~79,500 tokens  (79.5%)
```

If the budget is exceeded: compress older episodic memories via summarization before injecting.

## 5. Why This Matters

### Reliability
Each externalized component can be inspected, versioned, and corrected independently:
- Wrong user preference? → Update PERSONAL memory record
- Flawed trade procedure? → Fix the SKILL.md file in S3
- Broken API contract? → Update the MCP tool definition

### Consistency
The procedure for executing a trade is **always the same** — loaded from the same SKILL.md. Not re-derived from weights, which vary with temperature and prompt.

### Learnability
The episodic → semantic promotion means the agent gets better over time **without fine-tuning** — purely through structured experience accumulation.

### Human Control
Approval gates are explicit, auditable, and configurable. You can inspect every approval request and every decision in DynamoDB.

## 6. When to Use This Framework

**Good fit:**
- Agent that needs to remember user preferences across sessions
- Multi-step workflows with irreversible actions (financial, medical, legal)
- Domain expert agent with stable procedural knowledge
- Any agent calling external APIs that need structured error handling

**Poor fit:**
- Single-turn Q&A (no memory needed)
- Fully read-only agents (no approval gates needed)
- Rapid prototype / throwaway script (infrastructure overhead not worth it)

---

## 7. Real Implementation

This folder contains a full working implementation: **FinAgent** — a personal finance AI agent.

| Paper concept | FinAgent implementation |
|--------------|------------------------|
| Memory | `personal-finance-agent/memory/ddb_memory.py` |
| Skills | `personal-finance-agent/skills/*.md` |
| Protocols | MCP tools in `agent_core.py` (Alpaca, Polygon.io, Plaid, Experian) |
| Harness | `personal-finance-agent/agent/agent_core.py` — `FinAgent.invoke()` |
| Infrastructure | `personal-finance-agent/infrastructure/template.yaml` (AWS SAM) |

---

## Next: Phase 1
Open `phase1_basic_concepts.ipynb` to implement the memory system and skill loader from scratch.